In [8]:
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ihelon/coffee-sales")

print("Path to dataset files:", path)
print("Files:", os.listdir(path))

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

Path to dataset files: C:\Users\hieul\.cache\kagglehub\datasets\ihelon\coffee-sales\versions\21
Files: ['index_1.csv', 'index_2.csv']


In [21]:
df1 = pd.read_csv(path + "\\index_1.csv")
df2 = pd.read_csv(path + "\\index_2.csv")

# df = pd.concat([df1, df2], ignore_index=True)
# df["datetime"] = pd.to_datetime(df["datetime"])
# df = df.sort_values("datetime").reset_index(drop=True)

print(f"df1: {df1.shape}, df2: {df2.shape}")
df1.head()

df1: (3636, 6), df2: (262, 5)


,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 10:15:50.520,card,ANON-0000-0000-0001,38.7,Latte
1,2024-03-01,2024-03-01 12:19:22.539,card,ANON-0000-0000-0002,38.7,Hot Chocolate
2,2024-03-01,2024-03-01 12:20:18.089,card,ANON-0000-0000-0002,38.7,Hot Chocolate
3,2024-03-01,2024-03-01 13:46:33.006,card,ANON-0000-0000-0003,28.9,Americano
4,2024-03-01,2024-03-01 13:48:14.626,card,ANON-0000-0000-0004,38.7,Latte


In [ ]:
'''
Attendance coupon analysis
- Identify customers close to attendance threshold (median purchase/customer + 1)
- Calculate average sale value
- Estimate revenue gain if those customers make enough purchases to hit threshold
'''

# Attendance coupon analysis — df1 only (has card column)
card_counts = df1.dropna(subset=["card"]).groupby("card").size() # drop NA, count unique cards frequency

median_purchases = card_counts.median()
threshold = median_purchases + 1

print(f"Median purchases per card: {median_purchases}")
print(f"Attendance threshold: {threshold}")

# Customers within 1-2 purchases of the threshold but not alreayd there
nudgeable = card_counts[(card_counts >= threshold - 2) & (card_counts < threshold)]
print(f"\nNudgeable customers (1-2 below threshold): {len(nudgeable)}")

avg_sale = df1["money"].mean()
print(f"Average sale value: ${avg_sale:.2f}")

# Revenue estimate: each nudgeable customer buys enough to hit threshold
extra_purchases = (threshold - nudgeable).sum()
revenue_estimate = extra_purchases * avg_sale
print(f"\nEstimated extra purchases: {extra_purchases:.0f}")
print(f"Estimated revenue gain: ${revenue_estimate:.2f}")

Total unique cards: 1316
Median purchases per card: 1.0
Attendance threshold: 2.0

Nudgeable customers (1-2 below threshold): 771
Average sale value: $31.75

Estimated extra purchases: 771
Estimated revenue gain: $24476.83


In [29]:
# Time window discount analysis — use combined df for more data
df = pd.concat([df1, df2], ignore_index=True)
df["datetime"] = pd.to_datetime(df["datetime"], format="mixed")
df["hour"] = df["datetime"].dt.hour

hourly = df.groupby("hour")["money"].agg(["sum", "count"]).rename(columns={"sum": "revenue", "count": "sales"})

mean_sales = hourly["sales"].mean()
std_sales = hourly["sales"].std()

threshold_mean = mean_sales
threshold_1sd = mean_sales - std_sales

dead_hours_mean = hourly[hourly["sales"] < threshold_mean]
dead_hours_1sd = hourly[hourly["sales"] < threshold_1sd]

print(f"Mean sales/hour: {mean_sales:.1f},  1 SD below mean: {threshold_1sd:.1f}")
print(f"\nDead hours (below mean):  {dead_hours_mean.index.tolist()}")
print(f"Dead hours (below 1 SD):  {dead_hours_1sd.index.tolist()}")

# Cost of discount at various rates
discount_rates = [0.05, 0.10, 0.15]
for label, dead in [("Below mean", dead_hours_mean), ("Below 1 SD", dead_hours_1sd)]:
    at_risk = dead["revenue"].sum()
    print(f"\n{label} — revenue in dead hours: ${at_risk:.2f}")
    for rate in discount_rates:
        print(f"  {int(rate*100)}% discount cost: ${at_risk * rate:.2f}")

Mean sales/hour: 216.6,  1 SD below mean: 117.0

Dead hours (below mean):  [6, 7, 20, 21, 22, 23]
Dead hours (below 1 SD):  [6, 7, 23]

Below mean — revenue in dead hours: $19989.44
  5% discount cost: $999.47
  10% discount cost: $1998.94
  15% discount cost: $2998.42

Below 1 SD — revenue in dead hours: $3397.42
  5% discount cost: $169.87
  10% discount cost: $339.74
  15% discount cost: $509.61
